# FINALSCRAPING2 - Pipeline Tugas 1 (CNBC, 1 Sep 2021 s/d 1 Sep 2026)

**Isi notebook (end-to-end):**
1. Split URL -> Scraping (concurrent, resume) -> Merge 4 shard DB
2. Filter Geopolitik -> tabel `geopolitics_articles`
3. Preprocessing -> `data/cleaned/news_cleaned.csv`
4. Alignment -> `data/aligned/news_alignment.csv` + `data/aligned/daily_aligned_dataset.csv`

**Cara pakai:**
- `RUN_SCRAPING` dan `RUN_MERGE` default `False`: cell scraping/merge otomatis di-skip, aman dijalankan di mesin mana pun. Set `True` hanya di mesin yang punya `urls_shard_*.csv` / 4 shard DB.
- Jalankan Run All; output analisis masuk ke folder `data/` di root repo.
- Jangan commit file besar (`*.db`, `cnbc_articles*.csv`, `news_cleaned.csv`); pakai Drive. Yang di-commit: sampel + file kecil.


In [ ]:
import csv, glob, json, random, re, sqlite3, threading, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

SHARD_ID = 1          # <- ubah ke 1, 2, 3, atau 4
N_WORKERS = 8
RATE_START = 2.0      # req/s per orang (agregat tim = 4x; naikkan bila beda jaringan)
RATE_CAP = 5.0
MAX_ATTEMPTS = 3
COOLDOWN_SEC = 300
RUN_SCRAPING = False  # True hanya di mesin yang punya urls_shard_*.csv
RUN_MERGE = False     # True hanya saat menggabungkan 4 shard DB
DATA_DIR = Path("../data")

SITEMAP_CSV = "merged_sitemaps.csv"
SORTED_CSV = "urls_sorted.csv"
SHARD_CSV = f"urls_shard_{SHARD_ID}.csv"
DB_PATH = f"cnbc_articles_shard_{SHARD_ID}.db"
FAIL_CSV = f"failures_shard_{SHARD_ID}.csv"
BLOCK_CSV = f"blocks_shard_{SHARD_ID}.csv"
SUMMARY_JSON = f"summary_shard_{SHARD_ID}.json"

START = pd.Timestamp("2021-09-01", tz="UTC")
END = pd.Timestamp("2026-09-01", tz="UTC")

HEADERS = {
    "User-Agent": {
        1: "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
        2: "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0 Safari/537.36",
        3: "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36",
        4: "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36",
    }[SHARD_ID],
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
}


## 1. Split URL (otomatis, butuh `merged_sitemaps.csv`)

Filter tanggal terbit dari URL + bagi 4 bagian kontigu setelah diurutkan.


In [ ]:
if RUN_SCRAPING:
    def build_shards():
        df = pd.read_csv(SITEMAP_CSV).drop_duplicates("loc")
        df = df[df["loc"].str.endswith(".html")]
        pub = pd.to_datetime(df["loc"].str.extract(r"cnbc\.com/(\d{4}/\d{2}/\d{2})/")[0], utc=True, errors="coerce")
        df = df.assign(publish_date=pub).dropna(subset=["publish_date"])
        df = df[(df["publish_date"] >= START) & (df["publish_date"] <= END)].sort_values("publish_date").reset_index(drop=True)
        size = -(-len(df) // 4)
        df["shard"] = df.index // size + 1
        df[["loc", "publish_date", "shard"]].to_csv(SORTED_CSV, index=False)
        for s in range(1, 5):
            df.loc[df["shard"] == s, ["loc", "publish_date"]].to_csv(f"urls_shard_{s}.csv", index=False)
        print(f"total {len(df)} URL | per shard: {df['shard'].value_counts().sort_index().to_dict()}")


    if not Path(SHARD_CSV).exists():
        build_shards()
    else:
        print(f"{SHARD_CSV} sudah ada, lewati split. Total: {len(pd.read_csv(SHARD_CSV))}")
else:
    print("skip: RUN_SCRAPING=False (split URL)")


## 2. Parser (author string/list diperbaiki, status paywall ditandai)


In [ ]:
def parse_author(author):
    if isinstance(author, str):
        return author
    if isinstance(author, dict):
        return author.get("name", "")
    if isinstance(author, list):
        return "; ".join(a if isinstance(a, str) else a.get("name", "") for a in author if isinstance(a, (str, dict)))
    return ""


def extract_categories(html):
    cats = {}
    for raw in re.findall(r'\{"headline":"[^{}]*"__typename":"tag"\}', html):
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if obj.get("type") == "franchise":
            key = obj.get("id") or obj.get("headline")
            cats[key] = obj.get("tagName") or obj.get("headline")
    return list(cats.values())


def parse_article(url, html):
    soup = BeautifulSoup(html, "html.parser")
    ld = {}
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            obj = json.loads(script.string or "")
        except (json.JSONDecodeError, TypeError):
            continue
        if isinstance(obj, dict) and obj.get("@type") == "NewsArticle":
            ld = obj
            break

    bodies = soup.select("div[class*='ArticleBody']")
    body = max(bodies, key=lambda d: len(d.find_all("p")), default=None)
    paragraphs = [p.get_text(" ", strip=True) for p in (body.find_all("p") if body else [])]
    content = "\n".join(t for t in paragraphs if t and t != "In this article")

    return {
        "url": url,
        "title": ld.get("headline") or "",
        "author": parse_author(ld.get("author")),
        "date_published": ld.get("datePublished") or "",
        "date_modified": ld.get("dateModified") or "",
        "categories": extract_categories(html),
        "content": content,
        "content_status": "ok" if content else ("paywalled" if body else "missing"),
    }


BLOCK_MARKERS = ("access denied", "unusual traffic", "captcha", "are you a robot", "request blocked")


def looks_blocked(text):
    if len(text) > 50000:
        return False
    low = text.lower()
    return any(m in low for m in BLOCK_MARKERS)


## 3. Database + resume (jalankan ulang aman)


In [ ]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS articles (
    url TEXT PRIMARY KEY, title TEXT, author TEXT,
    date_published TEXT, date_modified TEXT, content TEXT,
    content_status TEXT, http_status INTEGER, fetched_at TEXT
);
CREATE TABLE IF NOT EXISTS article_categories (
    url TEXT, category TEXT, PRIMARY KEY (url, category)
);
CREATE TABLE IF NOT EXISTS scrape_failures (
    url TEXT PRIMARY KEY, attempts INTEGER, last_status INTEGER,
    error TEXT, last_attempt_at TEXT
);
"""


def open_db(path=DB_PATH):
    conn = sqlite3.connect(path)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.executescript(SCHEMA)
    conn.commit()
    return conn


if RUN_SCRAPING:
    conn = open_db()
    done = {r[0] for r in conn.execute("SELECT url FROM articles")}
    failed = {u: (a, s) for u, a, s in conn.execute("SELECT url, attempts, last_status FROM scrape_failures")}
    urls = pd.read_csv(SHARD_CSV)["loc"].tolist()
    pending = [
        u for u in urls
        if u not in done and failed.get(u, (0, 0))[0] < MAX_ATTEMPTS and failed.get(u, (0, 0))[1] not in (404, 410)
    ]
    random.seed(SHARD_ID)
    random.shuffle(pending)
    print(f"shard {SHARD_ID}: {len(urls)} URL | selesai {len(done)} | akan diskrape {len(pending)}")
else:
    print("skip: RUN_SCRAPING=False (database + resume)")


## 4. Runner (concurrent + rate adaptif + circuit breaker)


In [ ]:
if RUN_SCRAPING:
    class RateLimiter:
        def __init__(self, rate):
            self.rate = rate
            self.lock = threading.Lock()
            self.next_time = time.monotonic()

        def wait(self):
            with self.lock:
                now = time.monotonic()
                self.next_time = max(self.next_time, now)
                delay = self.next_time - now
                self.next_time += 1.0 / self.rate
            if delay > 0:
                time.sleep(delay)

        def slow_down(self):
            with self.lock:
                self.rate = max(0.5, self.rate / 2)

        def speed_up(self):
            with self.lock:
                self.rate = min(RATE_CAP, self.rate + 0.25)

        def force(self, rate):
            with self.lock:
                self.rate = rate


    limiter = RateLimiter(RATE_START)
    local = threading.local()

    try:
        COOKIES = requests.get("https://www.cnbc.com/", headers=HEADERS, timeout=15).cookies.get_dict()
        print("warm-up ok, cookies:", len(COOKIES))
    except Exception as e:
        COOKIES = {}
        print("warm-up gagal (lanjut tanpa cookie):", e)


    def get_session():
        if not hasattr(local, "session"):
            s = requests.Session()
            s.headers.update(HEADERS)
            if COOKIES:
                s.cookies.update(COOKIES)
            local.session = s
        return local.session


    def fetch(url):
        status, error = 0, ""
        for attempt in range(1, MAX_ATTEMPTS + 1):
            try:
                limiter.wait()
                r = get_session().get(url, timeout=20)
                status = r.status_code
                if status == 200:
                    if not looks_blocked(r.text):
                        return r.text, 200, ""
                    status, error = 0, "soft-block"
                elif status in (404, 410):
                    return None, status, "dead link"
                else:
                    error = f"http {status}"
                    retry_after = r.headers.get("Retry-After", "")
                    if retry_after.isdigit():
                        time.sleep(min(int(retry_after), 120))
            except Exception as e:
                status, error = 0, type(e).__name__
            limiter.slow_down()
            time.sleep(2 ** (attempt - 1) + random.random())
        return None, status, error


    def save_success(item, status):
        now = datetime.now(timezone.utc).isoformat()
        conn.execute(
            "INSERT OR REPLACE INTO articles (url, title, author, date_published, date_modified, "
            "content, content_status, http_status, fetched_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
            (item["url"], item["title"], item["author"], item["date_published"], item["date_modified"],
             item["content"], item["content_status"], status, now),
        )
        conn.execute("DELETE FROM article_categories WHERE url = ?", (item["url"],))
        for cat in item["categories"]:
            conn.execute("INSERT OR IGNORE INTO article_categories (url, category) VALUES (?, ?)", (item["url"], cat))
        conn.execute("DELETE FROM scrape_failures WHERE url = ?", (item["url"],))


    def save_failure(url, status, error):
        now = datetime.now(timezone.utc).isoformat()
        attempts = failed.get(url, (0, 0))[0] + 1
        failed[url] = (attempts, status)
        conn.execute(
            "INSERT INTO scrape_failures (url, attempts, last_status, error, last_attempt_at) VALUES (?, ?, ?, ?, ?) "
            "ON CONFLICT(url) DO UPDATE SET attempts=excluded.attempts, last_status=excluded.last_status, "
            "error=excluded.error, last_attempt_at=excluded.last_attempt_at",
            (url, attempts, status, error, now),
        )
        with open(FAIL_CSV, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([url, attempts, status, error, now])


    if not Path(FAIL_CSV).exists():
        with open(FAIL_CSV, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(["url", "attempts", "status", "error", "time"])
    if not Path(BLOCK_CSV).exists():
        with open(BLOCK_CSV, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(["time", "status", "error", "rate"])

    success, dead, fail, blocks, consecutive = 0, 0, 0, 0, 0
    content_counts = Counter()
    started = time.time()
    total = len(pending)

    with ThreadPoolExecutor(N_WORKERS) as pool:
        futures = {pool.submit(fetch, u): u for u in pending}
        for i, fut in enumerate(as_completed(futures), 1):
            url = futures[fut]
            try:
                html, status, error = fut.result()
            except Exception as e:
                html, status, error = None, 0, type(e).__name__
            if html is not None:
                article = parse_article(url, html)
                save_success(article, status)
                success += 1
                content_counts[article["content_status"]] += 1
                consecutive = 0
                if success % 500 == 0:
                    limiter.speed_up()
            elif status in (404, 410):
                save_failure(url, status, error)
                dead += 1
                consecutive = 0
            else:
                save_failure(url, status, error)
                fail += 1
                if status in (403, 429, 503) or error == "soft-block":
                    consecutive += 1
                    with open(BLOCK_CSV, "a", newline="", encoding="utf-8") as f:
                        csv.writer(f).writerow([datetime.now(timezone.utc).isoformat(), status, error, round(limiter.rate, 2)])
                else:
                    consecutive = 0
            if i % 25 == 0:
                conn.commit()
            if consecutive >= 5:
                blocks += 1
                limiter.force(0.5)
                print(f"!! indikasi block x{consecutive} -> cooldown {COOLDOWN_SEC}s, rate 0.5/s")
                time.sleep(COOLDOWN_SEC)
                consecutive = 0
            if i % 100 == 0:
                elapsed = time.time() - started
                print(f"[{i}/{total}] ok={success} dead={dead} fail={fail} | {i / elapsed:.1f} url/s | ETA {(total - i) / (i / elapsed) / 60:.0f} mnt")
            if blocks >= 3:
                print("!! 3x cooldown - stop. Jalankan ulang notebook nanti (auto-resume).")
                pool.shutdown(wait=False, cancel_futures=True)
                break

    conn.commit()
    summary = {
        "shard": SHARD_ID, "urls": len(urls), "already_done": len(done), "attempted": total,
        "success": success, "dead": dead, "failed": fail,
        "content_status": dict(content_counts), "block_cooldowns": blocks,
        "seconds": round(time.time() - started, 1),
    }
    with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    print(summary)
else:
    print("skip: RUN_SCRAPING=False (runner scraping)")


## 5. Merge & QA (jalankan setelah 4 shard selesai; pastikan cell 1 dan 3 sudah dijalankan)


In [ ]:
if RUN_MERGE:
    FINAL_DB = "cnbc_articles.db"
    main = open_db(FINAL_DB)

    for path in sorted(glob.glob("cnbc_articles_shard_*.db")):
        main.execute("ATTACH DATABASE ? AS shard", (path,))
        main.execute("INSERT OR IGNORE INTO articles SELECT * FROM shard.articles")
        main.execute("INSERT OR IGNORE INTO article_categories SELECT * FROM shard.article_categories")
        main.execute("INSERT OR IGNORE INTO scrape_failures SELECT * FROM shard.scrape_failures")
        main.commit()
        main.execute("DETACH DATABASE shard")

    df = pd.read_sql(
        "SELECT a.url, a.title, a.author, a.date_published, a.date_modified, "
        "group_concat(c.category, '; ') AS categories, a.content, a.content_status "
        "FROM articles a LEFT JOIN article_categories c ON a.url = c.url "
        "GROUP BY a.url ORDER BY a.date_published",
        main,
    )
    df.to_csv("cnbc_articles.csv", index=False, quoting=csv.QUOTE_ALL)

    fail_files = sorted(glob.glob("failures_shard_*.csv"))
    if fail_files:
        fails = pd.concat([pd.read_csv(p) for p in fail_files], ignore_index=True).drop_duplicates("url")
        fails.to_csv("failures_all.csv", index=False)
        fails[fails["status"].isin([404, 410])].to_csv("dead_links.csv", index=False)

    report = []
    for path in sorted(glob.glob("cnbc_articles_shard_*.db")):
        c = sqlite3.connect(path)
        report.append({
            "shard": path.split("shard_")[-1].split(".")[0],
            "articles": c.execute("SELECT count(*) FROM articles").fetchone()[0],
            "paywalled": c.execute("SELECT count(*) FROM articles WHERE content_status='paywalled'").fetchone()[0],
            "failures": c.execute("SELECT count(*) FROM scrape_failures").fetchone()[0],
        })
        c.close()
    report = pd.DataFrame(report)
    report.to_csv("scrape_report.csv", index=False)

    print(f"total artikel: {len(df)} | duplikat: {df['url'].duplicated().sum()}")
    print(df["content_status"].value_counts().to_string())
    print(report.to_string(index=False))
else:
    print("skip: RUN_MERGE=False (merge shard DB)")


## 6. Filter Geopolitik (Opsi 3 - hybrid, keyword KETAT)

Artikel masuk jika: punya kategori inti **Tier A** ATAU (judul mengandung keyword geopolitik DAN punya kategori politik/world), lalu hanya `content_status='ok'`.
Output: tabel `geopolitics_articles` + `cnbc_articles_geopolitics.csv` + `geopolitics_category_report.csv`. Dataset asli tidak diubah.


In [ ]:
GEOPOLITICS_CORE = [
    "Wars and Military Conflicts", "World Politics", "Europe Politics", "Asia Politics", "China Politics",
    "Defense", "Aerospace & Defense", "Defense Contractors", "Nuclear Weapons", "Terrorism",
    "Russia Stories", "Ukraine Stories", "Israel News", "Access Middle East", "European Union",
    "International Organizations", "The China Connection", "Middle East Money",
    "International: Top News And Analysis", "Global Economy: CNBC Explains", "Political Capital",
    "Markets and Politics Digital Original Video",
]
KEYWORD_CATEGORIES = [
    "World News", "Political Leaders", "International Homepage", "Europe: Top News And Analysis",
    "Asia: Top News and Analysis", "Europe Homepage", "Asia-Pacific Homepage", "Europe Early Edition",
    "Latin America", "Politics", "White House", "Congress", "Policy", "Elections",
    "Government Agencies", "United States Election 2024",
]
GEOPOL_KEYWORDS = re.compile(
    r"\b(?:russia|russian|ukraine|ukrainian|putin|zelensky|kremlin|china|chinese|taiwan|"
    r"north korea|south korea|kim jong|iran|israel|gaza|hamas|hezbollah|houthi|taliban|"
    r"nato|opec|brics|european union|united nations|g7|g20|tariffs?|trade war|sanctions?|"
    r"missiles?|military|troops|wars?|ceasefire|treaty|summit|diplomat\w*|foreign minister|"
    r"state department|pentagon|refugees?|nuclear weapons?|hostages?|airstrikes?|"
    r"geopolit\w*|foreign policy|embassy)\b",
    re.I,
)

GEO_DB = "cnbc_articles.db"
geo = open_db(GEO_DB)
articles = pd.read_sql(
    "SELECT url, title, author, date_published, date_modified, content, content_status FROM articles",
    geo,
)
cats = pd.read_sql("SELECT url, category FROM article_categories", geo)

core_map = cats[cats["category"].isin(GEOPOLITICS_CORE)].groupby("url")["category"].apply(list)
kwcat_map = cats[cats["category"].isin(KEYWORD_CATEGORIES)].groupby("url")["category"].apply(list)
title_kw = set(articles.loc[articles["title"].fillna("").str.contains(GEOPOL_KEYWORDS, regex=True), "url"])
core_urls = set(core_map.index)
kw_urls = set(kwcat_map.index) & title_kw

selected = articles[articles["url"].isin(core_urls | kw_urls) & (articles["content_status"] == "ok")].copy()
selected["matched_categories"] = selected["url"].map(lambda u: "; ".join(core_map.get(u) or kwcat_map.get(u, [])))
selected["match_rule"] = selected["url"].map(lambda u: "tier_a" if u in core_urls else "keyword")
selected = selected[["url", "title", "author", "date_published", "date_modified",
                     "content", "content_status", "matched_categories", "match_rule"]]

geo.execute("DROP TABLE IF EXISTS geopolitics_articles")
selected.to_sql("geopolitics_articles", geo, index=False)
geo.commit()
selected.to_csv("cnbc_articles_geopolitics.csv", index=False, quoting=csv.QUOTE_ALL)

total = cats.groupby("category").size().rename("articles_total")
subset = cats[cats["url"].isin(selected["url"])].groupby("category").size().rename("articles_geopolitics")
report = pd.concat([total, subset], axis=1).fillna(0).astype(int).reset_index()
report["tier"] = report["category"].map(
    lambda c: "core" if c in GEOPOLITICS_CORE else ("keyword" if c in KEYWORD_CATEGORIES else "other")
)
report.sort_values("articles_geopolitics", ascending=False).to_csv("geopolitics_category_report.csv", index=False)

print(f"kategori inti: {len(core_urls)} URL | keyword+politik: {len(kw_urls)} URL")
print(f"hasil akhir (konten ok): {len(selected)} artikel")
print(selected["match_rule"].value_counts().to_string())
print("CSV: cnbc_articles_geopolitics.csv | report: geopolitics_category_report.csv")
for _, r in selected.sample(min(12, len(selected)), random_state=7).iterrows():
    print(" *", r["title"][:105])
geo.execute("PRAGMA wal_checkpoint(TRUNCATE)")
geo.close()


## 7. Preprocessing (cleaning + dedup + filtering log)

Input: tabel `geopolitics_articles`. Cleaning light + buang boilerplate, dedup konten (rasio >= 0.99 dalam grup judul sama), buang artikel tanpa hari perdagangan berikutnya.
Output: `data/raw/news_raw_sample.csv`, `data/cleaned/news_cleaned.csv` (+sample), `data/metadata/{filtering_log,removed_articles,data_dictionary}.csv`.


In [ ]:
import hashlib, unicodedata
from difflib import SequenceMatcher

for sub in ("raw", "cleaned", "aligned", "metadata"):
    (DATA_DIR / sub).mkdir(parents=True, exist_ok=True)

rate_raw = pd.read_csv(DATA_DIR / "raw" / "exchange_rate_bi_raw.csv")
rate_raw["trading_date"] = pd.to_datetime(rate_raw["Tanggal"], format="%m/%d/%Y %I:%M:%S %p").dt.date
TRADING_DATES = sorted(rate_raw["trading_date"])
LAST_TRADING_DATE = TRADING_DATES[-1]

BOILERPLATE = [
    re.compile(r"^In this article$", re.I),
    re.compile(r"^Subscribe to CNBC on YouTube\.?$", re.I),
    re.compile(r"^WATCH:", re.I),
    re.compile(r"^[\u2014\u2013-]\s*.*contributed to this report", re.I),
    re.compile(r"^CNBC'?s?\s+.*contributed to this report", re.I),
    re.compile(r"^(Reuters|Associated Press|AP)\b.*contributed to this report", re.I),
    re.compile(r"^.*contributed to this report\s*(?:from\s+[^.]*)?\.?\s*$", re.I),
    re.compile(r"^Read more (from )?NBC News:?", re.I),
    re.compile(r"^https?://\S+$", re.I),
    re.compile(r"^Sign up here to receive .*newsletter\s*\.?\s*$", re.I),
    re.compile(r"^Disclosure: Comcast owns NBCUniversal", re.I),
]


def clean_text(text):
    lines = []
    for line in str(text).split("\n"):
        line = unicodedata.normalize("NFC", line).replace("\xa0", " ")
        line = re.sub(r"\s+", " ", line).strip()
        if line and not any(p.match(line) for p in BOILERPLATE):
            lines.append(line)
    return "\n".join(lines)


db = open_db("cnbc_articles.db")
total_articles = db.execute("SELECT count(*) FROM articles").fetchone()[0]
content_ok = db.execute("SELECT count(*) FROM articles WHERE content_status='ok'").fetchone()[0]
geo = pd.read_sql(
    "SELECT g.url, g.title, g.author, g.date_published, g.date_modified, g.content, "
    "g.matched_categories, g.match_rule, a.fetched_at "
    "FROM geopolitics_articles g LEFT JOIN articles a ON g.url = a.url",
    db,
)
db.close()

geo.head(500).to_csv(DATA_DIR / "raw" / "news_raw_sample.csv", index=False)

geo["article_id"] = "cnbc_" + geo["url"].map(lambda u: hashlib.md5(u.encode()).hexdigest()[:8])
geo["source"] = "CNBC"
geo["published_at_utc"] = pd.to_datetime(geo["date_published"], utc=True)
geo["published_at_wib"] = geo["published_at_utc"].dt.tz_convert("Asia/Jakarta")
geo["publication_date"] = geo["published_at_wib"].dt.date
geo["title_clean"] = geo["title"].map(clean_text)
geo["content_clean"] = geo["content"].map(clean_text)
geo["duplicate_group"] = ""

reasons = pd.Series("", index=geo.index)
for title, g in geo.groupby("title"):
    if len(g) < 2 or not str(title).strip():
        continue
    group_id = "dup_" + hashlib.md5(title.encode()).hexdigest()[:6]
    geo.loc[g.index, "duplicate_group"] = group_id
    base = re.sub(r"\s+", " ", unicodedata.normalize("NFC", g.iloc[0]["content"]).replace("\xa0", " ")).strip()
    for idx, row in g.iloc[1:].iterrows():
        other = re.sub(r"\s+", " ", unicodedata.normalize("NFC", row["content"]).replace("\xa0", " ")).strip()
        if SequenceMatcher(None, base, other).ratio() >= 0.99:
            reasons[idx] = "duplicate"

reasons[(geo["publication_date"] >= LAST_TRADING_DATE) & (reasons == "")] = "no_next_trading_day"
reasons[(geo["title_clean"].str.strip() == "") & (reasons == "")] = "missing_title"

geo["filter_status"] = "kept"
geo.loc[reasons != "", "filter_status"] = "removed"
geo["filter_reason"] = reasons

kept = geo[geo["filter_status"] == "kept"].sort_values("published_at_wib").copy()
removed = geo[geo["filter_status"] == "removed"].copy()
kept["scraped_at"] = kept["fetched_at"]
kept["published_at_utc"] = kept["published_at_utc"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")
kept["published_at_wib"] = kept["published_at_wib"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")

OUT_COLS = ["article_id", "source", "url", "title_clean", "content_clean", "published_at_utc",
            "published_at_wib", "match_rule", "matched_categories",
            "duplicate_group", "scraped_at"]
kept[OUT_COLS].to_csv(DATA_DIR / "cleaned" / "news_cleaned.csv", index=False)
kept[OUT_COLS].head(500).to_csv(DATA_DIR / "cleaned" / "news_cleaned_sample.csv", index=False)
removed[["article_id", "url", "title", "filter_reason"]].to_csv(DATA_DIR / "metadata" / "removed_articles.csv", index=False)

log = pd.DataFrame([
    ("raw_total_articles", total_articles),
    ("raw_content_ok", content_ok),
    ("geopolitics_selected", len(geo)),
    ("removed_duplicate", int((reasons == "duplicate").sum())),
    ("removed_missing_title", int((reasons == "missing_title").sum())),
    ("removed_no_next_trading_day", int((reasons == "no_next_trading_day").sum())),
    ("final_cleaned", len(kept)),
], columns=["stage", "articles"])
log.to_csv(DATA_DIR / "metadata" / "filtering_log.csv", index=False)

dictionary = pd.DataFrame([
    ("news_cleaned.csv", "article_id", "ID unik cnbc_<8 digit MD5(url)>"),
    ("news_cleaned.csv", "title_clean", "Judul setelah normalisasi unicode + buang boilerplate"),
    ("news_cleaned.csv", "content_clean", "Isi setelah normalisasi unicode + buang boilerplate"),
    ("news_cleaned.csv", "published_at_utc", "Waktu publikasi asli CNBC (UTC)"),
    ("news_cleaned.csv", "published_at_wib", "Waktu publikasi dikonversi ke WIB"),
    ("news_cleaned.csv", "match_rule", "tier_a (kategori inti) atau keyword (judul + kategori politik)"),
    ("news_cleaned.csv", "duplicate_group", "ID grup judul duplikat (kosong bila unik)"),
    ("news_cleaned.csv", "scraped_at", "Waktu artikel di-scrape"),
    ("exchange_rate_bi_cleaned.csv", "trading_date", "Tanggal perdagangan (kalender dari data BI)"),
    ("exchange_rate_bi_cleaned.csv", "usd_idr_rate", "Nilai Rupiah untuk 1 USD (JISDOR)"),
    ("exchange_rate_bi_cleaned.csv", "rate_type", "Jenis kurs: JISDOR"),
    ("news_alignment.csv", "day_type", "trading_day / weekend / holiday"),
    ("news_alignment.csv", "effective_trading_date", "Hari perdagangan pertama setelah publikasi WIB"),
    ("news_alignment.csv", "alignment_rule", "Aturan pemetaan: next_trading_day"),
    ("daily_aligned_dataset.csv", "usd_idr_rate", "Kurs JISDOR hari perdagangan"),
    ("daily_aligned_dataset.csv", "previous_rate", "Kurs hari perdagangan sebelumnya"),
    ("daily_aligned_dataset.csv", "rate_change", "Selisih kurs terhadap hari sebelumnya"),
    ("daily_aligned_dataset.csv", "daily_return", "rate_change / previous_rate"),
    ("daily_aligned_dataset.csv", "geopolitical_news_count", "Jumlah berita geopolitik yang dipetakan ke hari itu"),
    ("daily_aligned_dataset.csv", "article_ids", "Daftar article_id (dipisah koma)"),
    ("daily_aligned_dataset.csv", "has_geopolitical_news", "True bila ada berita geopolitik"),
], columns=["file", "column", "description"])
dictionary.to_csv(DATA_DIR / "metadata" / "data_dictionary.csv", index=False)

print(f"input geopolitik: {len(geo)} | kept: {len(kept)} | removed: {len(removed)}")
print(log.to_string(index=False))
print("output: data/cleaned/news_cleaned.csv, data/metadata/*.csv")


## 8. Alignment (berita -> hari perdagangan, next trading day)

Aturan: kurs JISDOR terbit tengah malam untuk hari itu, jadi berita tanggal t dipetakan ke hari perdagangan pertama SETELAH t. Kalender hari perdagangan = tanggal di `exchange_rate_bi_raw.csv`.
Output: `data/cleaned/exchange_rate_bi_cleaned.csv`, `data/aligned/news_alignment.csv`, `data/aligned/daily_aligned_dataset.csv`.


In [ ]:
import numpy as np

rate_raw = pd.read_csv(DATA_DIR / "raw" / "exchange_rate_bi_raw.csv")
rate_raw["trading_date"] = pd.to_datetime(rate_raw["Tanggal"], format="%m/%d/%Y %I:%M:%S %p")
rate = (
    rate_raw.sort_values("trading_date")[["trading_date", "Kurs"]]
    .rename(columns={"Kurs": "usd_idr_rate"})
    .reset_index(drop=True)
)
rate["trading_date"] = rate["trading_date"].dt.date
rate["currency"] = "USD"
rate["rate_type"] = "JISDOR"
rate["source"] = "Bank Indonesia"
rate["is_trading_day"] = True
rate["retrieved_at"] = "2026-09-04"

trading = np.array(sorted(rate["trading_date"]), dtype="datetime64[D]")
clean = pd.read_csv(DATA_DIR / "cleaned" / "news_cleaned.csv")
pub = pd.to_datetime(clean["published_at_utc"], utc=True).dt.tz_convert("Asia/Jakarta")
pub_dates = pub.dt.date
idx = np.searchsorted(trading, np.array(pub_dates, dtype="datetime64[D]"), side="right")
if (idx >= len(trading)).any():
    raise ValueError("ada artikel tanpa hari perdagangan berikutnya")

effective = pd.to_datetime(trading[idx])
pub_in_calendar = pd.Series(pub_dates).isin(set(rate["trading_date"])).values
alignment = pd.DataFrame({
    "article_id": clean["article_id"].values,
    "published_at_wib": pub.dt.strftime("%Y-%m-%d %H:%M:%S%z").values,
    "publication_date": pub.dt.strftime("%Y-%m-%d").values,
    "day_type": np.where(pub.dt.weekday >= 5, "weekend", np.where(pub_in_calendar, "trading_day", "holiday")),
    "effective_trading_date": effective.strftime("%Y-%m-%d"),
    "alignment_rule": "next_trading_day",
    "next_trading_date": effective.strftime("%Y-%m-%d"),
})

grouped = alignment.groupby("effective_trading_date")["article_id"].apply(list)
daily = rate.copy()
daily["trading_date"] = daily["trading_date"].astype(str)
daily["previous_rate"] = daily["usd_idr_rate"].shift(1)
daily["rate_change"] = daily["usd_idr_rate"] - daily["previous_rate"]
daily["daily_return"] = daily["rate_change"] / daily["previous_rate"]
daily["geopolitical_news_count"] = daily["trading_date"].map(grouped).apply(lambda v: len(v) if isinstance(v, list) else 0)
daily["article_ids"] = daily["trading_date"].map(grouped).apply(lambda v: ",".join(v) if isinstance(v, list) else "")
daily["has_geopolitical_news"] = daily["geopolitical_news_count"] > 0
daily = daily[["trading_date", "usd_idr_rate", "previous_rate", "rate_change", "daily_return",
               "geopolitical_news_count", "article_ids", "has_geopolitical_news"]]

assert len(daily) == len(rate)
assert daily["usd_idr_rate"].notna().all()
assert int(daily["geopolitical_news_count"].sum()) == len(clean)
assert (effective.values > np.array(pub_dates, dtype="datetime64[ns]")).all()

rate.to_csv(DATA_DIR / "cleaned" / "exchange_rate_bi_cleaned.csv", index=False)
alignment.to_csv(DATA_DIR / "aligned" / "news_alignment.csv", index=False)
daily.to_csv(DATA_DIR / "aligned" / "daily_aligned_dataset.csv", index=False)

print(f"trading days: {len(rate)} | artikel terpetakan: {len(clean)}")
print(f"hari dengan berita: {(daily['geopolitical_news_count'] > 0).sum()} | maksimum {int(daily['geopolitical_news_count'].max())} berita/hari")
print("day_type:", alignment["day_type"].value_counts().to_dict())
print(daily.head(3).to_string(index=False))
print("output: data/cleaned/exchange_rate_bi_cleaned.csv, data/aligned/*.csv")
